# Chatbot vs human agent tickets: Brunner-Munzel test

**Brunner-Munzel Test** · a worked example from the StatNex teaching guide

**The story.** An online shop answers support tickets two ways: a chatbot (18 tickets) and a human agent (9 tickets). It recorded the minutes needed to resolve each ticket. The chatbot's times are very spread out (from 3 to 70 minutes), while the human agent's times are tightly bunched between 11 and 19 minutes.

**The question.** Is a chatbot ticket more likely than not to be faster, even though the two groups have very different spreads?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/brunner_munzel.csv` (18 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Chatbot,Human agent
5.0,14.0
30.0,12.0
4.0,16.0
7.0,13.0
6.0,19.0
42.0,11.0
3.0,15.0
5.0,17.0
9.0,13.0
6.0,
55.0,
4.0,
8.0,
7.0,
6.0,
5.0,
9.0,
70.0,

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One column per sample; text and blank cells are skipped, so the samples may differ
# in size. Every column of sample 1 is tested against every column of sample 2.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
samples1 = ["Chatbot"]
samples2 = ["Human agent"]
values = {name: pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
          for name in [*samples1, *samples2]}

## Brunner-Munzel test

In [ ]:
# The question: if you draw one value from each sample, is the sample 2 value larger
# more often than not? The relative effect p = P(X < Y) + P(X = Y) / 2 is that chance
# (a tie counts half), and the test asks whether p = 1/2. Unlike Mann-Whitney it does
# not assume the two samples have the same shape or spread: each sample's rank
# variance is estimated separately, as Welch does for the t test.
# scipy.stats.brunnermunzel gives the statistic and both P values. Its W is Brunner and
# Munzel's, the app's too: the relative effect minus 1/2 over its standard error, so a
# positive W means sample 2 tends to be larger. scipy does not return the degrees of
# freedom of its t approximation, so they are written out below.
mean_ranks, tests = [], []
for a in samples1:
    for b in samples2:
        x, y = values[a], values[b]
        n1, n2 = len(x), len(y)
        ranks = stats.rankdata(np.concatenate([x, y]))
        rx, ry = ranks[:n1], ranks[n1:]
        mean_ranks += [[a, n1, rx.mean()], [b, n2, ry.mean()]]
        # Each sample's rank variance, after taking out its ranking within itself.
        var_x = ((rx - stats.rankdata(x) - rx.mean() + (n1 + 1) / 2) ** 2).sum() / (n1 - 1)
        var_y = ((ry - stats.rankdata(y) - ry.mean() + (n2 + 1) / 2) ** 2).sum() / (n2 - 1)
        # Welch-Satterthwaite degrees of freedom for the t approximation.
        df = (n1 * var_x + n2 * var_y) ** 2 / ((n1 * var_x) ** 2 / (n1 - 1) + (n2 * var_y) ** 2 / (n2 - 1))
        # The normal curve suits large samples; the t curve with df degrees of freedom is
        # the one to read when either sample is small.
        normal = stats.brunnermunzel(x, y, distribution="normal")
        small = stats.brunnermunzel(x, y, distribution="t")
        w = small.statistic
        p_normal, p_t = normal.pvalue, small.pvalue
        # The relative effect itself, from the mean ranks: the share of (sample 1, sample 2)
        # pairs in which the sample 2 value is larger, ties counting half.
        relative_effect = (ry.mean() - rx.mean()) / (n1 + n2) + 0.5
        print(f"{a} vs {b}: relative effect P({a} < {b}) + P(=)/2 = {relative_effect:.4f}")
        tests += [[a, b, "normal approximation", w, np.nan, p_normal, stars(p_normal)],
                  [a, b, "small sample (t)", w, df, p_t, stars(p_t)]]

mean_ranks = pd.DataFrame(mean_ranks, columns=["sample", "n", "mean rank"])
tests = pd.DataFrame(tests, columns=["sample 1", "sample 2", "method", "W", "df", "P", ""])
print(mean_ranks.to_string(index=False))
print()
print(tests.to_string(index=False))

## What the result says

The Brunner-Munzel test asks whether a ticket from one channel tends to be solved before a ticket from the other. It works on ranks, the positions of the times when all tickets are sorted from fastest to slowest. It does not need the two groups to have the same spread, which matters here because the chatbot's times are far more spread out.

The mean rank is 11.5 for the 18 chatbot tickets and 19 for the 9 human agent tickets. A lower rank means a shorter time. The statistic W is 2.75. The small sample version, which suits groups this small, gives a two-sided P value of 0.014. The P value is the chance of a gap this large if neither channel were faster.

So a chatbot ticket is more likely than not to be solved first. The test uses only the order of the times, so it does not show how slow the few very slow chatbot tickets are: the chatbot is faster, but less predictable.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.